# E1a + E1b — YOLOv5s Baseline Experiments

**E1a:** YOLOv5s @ 640px — one-stage baseline at standard resolution
**E1b:** YOLOv5s @ 1280px — resolution ablation (2× input size)

## Research question (E1a vs E1b)
Is small-detection performance bottleneck architectural (detector) or pipeline (input resolution)?
- Keep detector constant (YOLOv5s)
- Vary only input size: 640px (E1a) vs 1280px (E1b)
- Compare mAP — especially AP_small

## Workflow
1. Load experiment configs
2. **Pipeline test** on `debug_500` subset (fast validation)
3. **E1a:** train YOLOv5s @ 640px → evaluate → save metrics
4. **E1b:** train YOLOv5s @ 1280px → evaluate → save metrics
5. **Compare:** side-by-side table, per-class AP heatmap, PR curves, sample predictions

In [1]:
import sys, os, json, yaml
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Project root
sys.path.append('..')

from src.env import get_device, get_paths, get_env_info, seed_everything
from src.datasets import visdrone_to_coco_json, VISDRONE_CLASSES
from src.metrics import compute_map, compute_pr_curve
from src.visualization import (set_plot_style, draw_bboxes, plot_pr_curve,
                               plot_per_class_ap_heatmap, plot_summary_table)
from src.trainers.yolo_trainer import YOLOTrainer

seed_everything(42)
paths = get_paths()
device = get_device()
set_plot_style()

print(json.dumps(get_env_info(), indent=2))
print(f"\nData path: {paths['data']}")
print(f"Output path: {paths['output']}")
print(f"Configs path: {paths['configs']}")

{
  "env": "local",
  "device": "mps",
  "cuda_available": false,
  "mps_available": true,
  "torch_version": "2.2.2"
}

Data path: /Users/davide/Desktop/Universita/AI/CV_DL/project/data/VisDrone2019-DET
Output path: /Users/davide/Desktop/Universita/AI/CV_DL/project/results/runs
Configs path: /Users/davide/Desktop/Universita/AI/CV_DL/project/configs


In [2]:
# Load experiment configs
config_dir = paths["configs"]
with open(config_dir / "e1a_yolo_640.yaml") as f:
    config_e1a = yaml.safe_load(f)
with open(config_dir / "e1b_yolo_1280.yaml") as f:
    config_e1b = yaml.safe_load(f)

print("E1a config:")
print(json.dumps(config_e1a, indent=2))
print("\nE1b config:")
print(json.dumps(config_e1b, indent=2))

# Check debug subset for pipeline test
debug_dir = paths["subsets"] / "debug_500"
debug_imgs = list(debug_dir.glob("images/*.jpg"))
print(f"\nDebug subset ({debug_dir.name}): {len(debug_imgs)} images available")

E1a config:
{
  "experiment": "e1a",
  "model": "yolov5s",
  "imgsz": 640,
  "epochs": 50,
  "batch_size": 8,
  "optimizer": "AdamW",
  "lr0": 0.001,
  "lrf": 0.01,
  "weight_decay": 0.0005,
  "warmup_epochs": 3,
  "patience": 10,
  "augment": {
    "mosaic": 1.0,
    "fliplr": 0.5,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "scale": 0.5
  },
  "seed": 42,
  "device": "auto"
}

E1b config:
{
  "experiment": "e1b",
  "model": "yolov5s",
  "imgsz": 1280,
  "epochs": 50,
  "batch_size": 4,
  "optimizer": "AdamW",
  "lr0": 0.001,
  "lrf": 0.01,
  "weight_decay": 0.0005,
  "warmup_epochs": 3,
  "patience": 10,
  "augment": {
    "mosaic": 1.0,
    "fliplr": 0.5,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "scale": 0.5
  },
  "seed": 42,
  "device": "auto"
}

Debug subset (debug_500): 500 images available


## Pipeline test — debug_500 subset

Validate entire pipeline works end-to-end before launching full training.
Runs 2 epochs on 500-image debug subset.

In [3]:
# Quick pipeline validation on debug_500 (2 epochs)
# Set force_retrain=False to skip if a valid run already exists.
FORCE_RETRAIN = False

config_debug = dict(config_e1a)
config_debug["epochs"] = 2
config_debug["experiment"] = f"{config_e1a['experiment']}_debug"   # → "e1a_debug"
config_debug["use_debug_subset"] = True
config_debug["force_retrain"] = FORCE_RETRAIN

print("=" * 60)
print(f"Pipeline test: YOLOv5s @ 640px on debug_500 (2 epochs)")
print(f"     output: {paths['output'] / config_debug['experiment']}")
print("=" * 60)

trainer_debug = YOLOTrainer(config_debug, paths)
trainer_debug.train()

# Load test metrics
test_metrics_path = trainer_debug.output_dir / "test_metrics.json"
if test_metrics_path.exists():
    with open(test_metrics_path) as f:
        debug_metrics = json.load(f)
    print(f"\nPipeline test metrics:")
    for k, v in debug_metrics.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
print(f"\n[OK] Pipeline validation complete. Checkpoints in {trainer_debug.output_dir / 'checkpoints'}")

Pipeline test: YOLOv5s @ 640px on debug_500 (2 epochs)
     output: /Users/davide/Desktop/Universita/AI/CV_DL/project/results/runs/e1a_debug
[SKIP] e1a_debug already completed. Set force_retrain=True to retrain.

Pipeline test metrics:
  mAP@0.5:0.95: 0.0000
  mAP@0.5: 0.0000
  mAP@0.75: 0.0000
  AP_small: 0.0000
  AP_medium: 0.0000
  AP_large: 0.0000

[OK] Pipeline validation complete. Checkpoints in /Users/davide/Desktop/Universita/AI/CV_DL/project/results/runs/e1a_debug/checkpoints


## E1a — YOLOv5s @ 640px

Training on full dataset (6,471 images), 50 epochs, batch=8, AdamW.

In [4]:
# E1a: YOLOv5s @ 640px
# Set FORCE_RETRAIN = True above to force retraining even if a run exists.
config_e1a["force_retrain"] = FORCE_RETRAIN

print("=" * 60)
print("E1a: YOLOv5s @ 640px — full training")
print(f"     epochs={config_e1a['epochs']}, batch={config_e1a['batch_size']}")
print("=" * 60)

trainer_e1a = YOLOTrainer(config_e1a, paths)
trainer_e1a.train()

E1a: YOLOv5s @ 640px — full training
     epochs=50, batch=8
[SKIP] e1a already completed. Set force_retrain=True to retrain.


{'mAP@0.5:0.95': 0.10276936092841593,
 'mAP@0.5': 0.19149496734606014,
 'mAP@0.75': 0.0919278070117098,
 'AP_small': 0.0,
 'AP_medium': 0.0,
 'AP_large': 0.0,
 'AP_per_class': [0.12642816971440668,
  0.16490970296413782,
  0.10392969120371405,
  0.23646841079547404,
  0.06570216815349131,
  0.04749900576936963,
  0.06935981775501847,
  0.03795319895530326,
  0.013801114015765236,
  0.16164232995747901]}

In [5]:
# Load E1a test metrics
with open(trainer_e1a.output_dir / "test_metrics.json") as f:
    metrics_e1a = json.load(f)

print("E1a final metrics:")
for k, v in metrics_e1a.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    elif isinstance(v, list):
        print(f"  {k}: [{', '.join(f'{x:.3f}' for x in v)}]")

# Save per-class AP as separate figure
ap_dict_e1a = {"E1a (640px)": metrics_e1a["AP_per_class"]}
plot_per_class_ap_heatmap(
    ap_dict_e1a,
    save_path=trainer_e1a.output_dir / "figures" / "per_class_ap.png",
)

E1a final metrics:
  mAP@0.5:0.95: 0.1028
  mAP@0.5: 0.1915
  mAP@0.75: 0.0919
  AP_small: 0.0000
  AP_medium: 0.0000
  AP_large: 0.0000
  AP_per_class: [0.126, 0.165, 0.104, 0.236, 0.066, 0.047, 0.069, 0.038, 0.014, 0.162]


/Users/davide/Desktop/Universita/AI/CV_DL/project/notebooks/../src/visualization.py:158: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## E1b — YOLOv5s @ 1280px

Resolution ablation: same model, 2× input size. Expect higher AP_small at cost of ~4× FLOPs.
Training: 50 epochs, batch=4 (memory constraint), AdamW.

In [ ]:
# E1b: YOLOv5s @ 1280px
# Set FORCE_RETRAIN = True above to force retraining even if a run exists.
config_e1b["force_retrain"] = FORCE_RETRAIN

print("=" * 60)
print("E1b: YOLOv5s @ 1280px — resolution ablation")
print(f"     epochs={config_e1b['epochs']}, batch={config_e1b['batch_size']}")
print("=" * 60)

trainer_e1b = YOLOTrainer(config_e1b, paths)
trainer_e1b.train()

In [ ]:
# Load E1b test metrics
with open(trainer_e1b.output_dir / "test_metrics.json") as f:
    metrics_e1b = json.load(f)

print("E1b final metrics:")
for k, v in metrics_e1b.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    elif isinstance(v, list):
        print(f"  {k}: [{', '.join(f'{x:.3f}' for x in v)}]")

plot_per_class_ap_heatmap(
    {"E1b (1280px)": metrics_e1b["AP_per_class"]},
    save_path=trainer_e1b.output_dir / "figures" / "per_class_ap.png",
)

## Comparison: E1a (640px) vs E1b (1280px)

Key question: does increasing input resolution improve small object detection?

In [ ]:
# Build comparison table
comparison = {
    "E1a (640px)": metrics_e1a,
    "E1b (1280px)": metrics_e1b,
}

# Filter to scalar metrics for table display
scalar_keys = [k for k in metrics_e1a if isinstance(metrics_e1a[k], (int, float))]
print("Comparison table:\n")
print(f"{'Metric':<20} {'E1a (640px)':<15} {'E1b (1280px)':<15} {'Δ':<10}")
print("-" * 60)
for k in scalar_keys:
    v1 = metrics_e1a.get(k, 0)
    v2 = metrics_e1b.get(k, 0)
    delta = v2 - v1
    print(f"{k:<20} {v1:<15.4f} {v2:<15.4f} {delta:<+10.4f}")

# Save comparison table as figure
output_dir = paths["output"]
plot_summary_table(
    comparison,
    save_path=output_dir / "e1a" / "figures" / "comparison_table.png",
)

In [ ]:
# Per-class AP comparison heatmap
ap_dict = {
    "E1a (640px)": metrics_e1a["AP_per_class"],
    "E1b (1280px)": metrics_e1b["AP_per_class"],
}

plot_per_class_ap_heatmap(
    ap_dict,
    save_path=output_dir / "e1a" / "figures" / "comparison_per_class_ap.png",
)

# Per-class bar chart
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(VISDRONE_CLASSES))
width = 0.35
ax.bar(x - width/2, ap_dict["E1a (640px)"], width, label="E1a (640px)", alpha=0.8)
ax.bar(x + width/2, ap_dict["E1b (1280px)"], width, label="E1b (1280px)", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(VISDRONE_CLASSES, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("AP")
ax.set_title("Per-class AP: E1a vs E1b")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / "e1a" / "figures" / "comparison_per_class_bar.png", dpi=150)
plt.show()

In [ ]:
# Per-class AP comparison — bar chart side-by-side
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(VISDRONE_CLASSES))
width = 0.35
ax.bar(x - width/2, metrics_e1a["AP_per_class"], width,
       label="E1a (640px)", color="steelblue", alpha=0.85)
ax.bar(x + width/2, metrics_e1b["AP_per_class"], width,
       label="E1b (1280px)", color="coral", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(VISDRONE_CLASSES, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("AP")
ax.set_title("Per-class AP: E1a (640px) vs E1b (1280px)")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

# Annotate best values
for i in range(10):
    val_a = metrics_e1a["AP_per_class"][i]
    val_b = metrics_e1b["AP_per_class"][i]
    best_val = max(val_a, val_b)
    best_x = x[i] - width/2 if val_a > val_b else x[i] + width/2
    ax.annotate(f"{best_val:.3f}", (best_x, best_val),
                ha="center", va="bottom", fontsize=7, fontweight="bold")

plt.tight_layout()
plt.savefig(output_dir / "e1a" / "figures" / "comparison_per_class_bar.png", dpi=150)
plt.show()

# Check which classes benefit most from higher resolution
print("Class-level AP delta (E1b - E1a):")
deltas = [b - a for a, b in zip(metrics_e1a["AP_per_class"], metrics_e1b["AP_per_class"])]
for cls_name, delta in zip(VISDRONE_CLASSES, deltas):
    arrow = "↑" if delta > 0 else "↓"
    print(f"  {cls_name:20s}: {delta:+.4f} {arrow}")

In [ ]:
# Sample predictions — visual comparison on validation images
from PIL import Image
from ultralytics import YOLO
import random

val_img_dir = paths["data"] / "images" / "val"
val_ann_dir = paths["data"] / "annotations" / "val"

# Load best models for prediction
model_e1a = YOLO(str(trainer_e1a.output_dir / "checkpoints" / "best.pt"))
model_e1b = YOLO(str(trainer_e1b.output_dir / "checkpoints" / "best.pt"))

# Pick a few sample images
sample_images = sorted(val_img_dir.glob("*.jpg"))
random.seed(42)
samples = random.sample(sample_images, min(4, len(sample_images)))

# 4 samples × 3 columns (GT, E1a, E1b) = 12 subplots → 4 rows × 3 cols
fig, axes = plt.subplots(len(samples), 3, figsize=(18, 5 * len(samples)))
if len(samples) == 1:
    axes = axes.reshape(1, -1)

for idx, img_path in enumerate(samples):
    img = np.array(Image.open(img_path).convert("RGB"))
    
    # Ground truth
    stem = img_path.stem
    ann_path = val_ann_dir / f"{stem}.txt"
    gt_boxes, gt_labels = [], []
    if ann_path.exists():
        with open(ann_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    h, w = img.shape[:2]
                    cx = float(parts[1]) * w
                    cy = float(parts[2]) * h
                    bw = float(parts[3]) * w
                    bh = float(parts[4]) * h
                    gt_boxes.append([cx - bw/2, cy - bh/2, cx + bw/2, cy + bh/2])
                    gt_labels.append(int(parts[0]))
    
    gt_img = draw_bboxes(img.copy(), gt_boxes, gt_labels)
    axes[idx, 0].imshow(gt_img)
    axes[idx, 0].set_title(f"GT — {img_path.name}")
    axes[idx, 0].axis("off")
    
    # E1a predictions
    results_a = model_e1a(img_path)[0]
    pred_boxes_a = results_a.boxes.xyxy.cpu().numpy()
    pred_scores_a = results_a.boxes.conf.cpu().numpy()
    pred_labels_a = results_a.boxes.cls.cpu().numpy().astype(int)
    pred_img_a = draw_bboxes(img.copy(), pred_boxes_a[:50], pred_labels_a[:50], pred_scores_a[:50])
    axes[idx, 1].imshow(pred_img_a)
    axes[idx, 1].set_title(f"E1a (640px) — {len(pred_boxes_a)} dets")
    axes[idx, 1].axis("off")
    
    # E1b predictions
    results_b = model_e1b(img_path)[0]
    pred_boxes_b = results_b.boxes.xyxy.cpu().numpy()
    pred_scores_b = results_b.boxes.conf.cpu().numpy()
    pred_labels_b = results_b.boxes.cls.cpu().numpy().astype(int)
    pred_img_b = draw_bboxes(img.copy(), pred_boxes_b[:50], pred_labels_b[:50], pred_scores_b[:50])
    axes[idx, 2].imshow(pred_img_b)
    axes[idx, 2].set_title(f"E1b (1280px) — {len(pred_boxes_b)} dets")
    axes[idx, 2].axis("off")

plt.tight_layout()
plt.savefig(output_dir / "e1a" / "figures" / "sample_predictions.png", dpi=150)
plt.show()

## Summary

### Key findings
- **E1a @ 640px**: baseline performance for YOLOv5s at standard resolution
- **E1b @ 1280px**: 2× resolution → expected improvement in AP_small
- Comparison saved to `results/runs/e1a/figures/comparison_*.png`

### Metrics saved
- `results/runs/e1a/metrics.jsonl` — per-epoch metrics
- `results/runs/e1a/test_metrics.json` — final validation metrics
- `results/runs/e1a/checkpoints/best.pt` — best checkpoint
- `results/runs/e1b/` — same structure for 1280px experiment

### Next steps
- **E2a + E2b**: Faster R-CNN (no FPN / +FPN) → `notebooks/02_faster_rcnn.ipynb`
- **E3**: SAHI tiling → `notebooks/03_sahi_tiling.ipynb`
- **E4**: Error analysis → `notebooks/04_analysis.ipynb`